In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from neuralhydrology.nh_run import eval_run
import pickle
from pathlib import Path
import os
from tqdm import tqdm


class Evaluation():
    
    """
    A class for evaluating a model trained by `Neuralhydrology` library
    -------------------
    Requirements:
        - A proper .txt file listing test basin id
        - A proper configuration (.yml) file specifing the test file and date
        - A folder consisting of .csv files of the test basin data with proper observed values
        
    -------------------
    Model Parameters:
        - run_dir: str
            Name of the directory for the target model
            
        - epoch_num: int
            The number of epoch to be evaluated
            
        - csv_dir: str
            The directory of the original csv files
            
        - eval_list:
            The name of the .txt file consisting of the basin list to be evaluated
            
        - mean: float
            A mean value used for standardizing the target variable
            
        - var: float
            A variance value used for standardizing the target variable
            
        - test_start_date: str
            Date to start evaluation 'dd/mm/yyyy`
            
        - test_end_date: str
            Date to end evaluation 'dd/mm/yyyy'
            
        - skip_sim: bool
            Whether to skip simulation - default to false, enable this option when simulation is already done
            
        - apply_trainsformation: bool
            Whether to apply inverse transformation - default to true, enable this when the standardization is not applied to the data
            
        - target_var: str
            Name of the target variable - default to "discharge"
    
    -------------------   
    Member Functions:
        public:
        - plot_validation() -> None:
            plot the validation error progress during training
            
        - collect_validation -> pd.DataFrame:
            returns a dataframe of validation errors for each epoch
            
        - get_single_pred(basin_id: str) -> float:
            returns the simulated nse value for the specified basin
            
        - get_pred(basin_list: str = test.txt) -> pd.DataFrame:
            returns a dataframe of the simulated nse values for the listed basins
            
        - plot_prediction(basin_id: str, skip_sim = skip_sim) -> None: 
            plot the simulated vs observed discharge (or target) values for a single basin
            
        - plot_predictions_distribution(basin_list: str = test.txt, skip_sim = self.skip_sim, ignore_neg = True) -> None:
            plot the distribution of nse values over multiple basins, excluding/not excluding negative nse values
            
        private:
        - __evalute_single__() -> float:
            evaluates a single basin and returns nse value
            
        - __evaluate__() -> pd.DataFrame:
            runs evaluation on the basins listed in eval_list and returns the result dataframe
    """
    
    def __init__(self, run_dir: str, 
                 epoch_num: int, 
                 csv_dir: str = "data/csv_files",
                 eval_list: str = r"basin_list\US_basin_list.txt",
                 mean: float = 0.8561527661255196,
                 var: float = 5.06157279557463,
                 test_start_date: str = '01/01/2011',
                 test_end_date: str = '31/12/2022',
                 skip_sim: bool = False,
                 apply_transformation: bool = True,
                 target_var: str = "discharge"
                 ):
        
        self.run_dir = run_dir
        self.epoch_num = epoch_num
        self.csv_dir = csv_dir
        self.eval_list = eval_list
        self.mean = mean
        self.var = var
        self.test_start_date = pd.to_datetime(test_start_date, format='%d/%m/%Y')
        self.test_end_date = pd.to_datetime(test_end_date, format='%d/%m/%Y')
        self.skip_sim = skip_sim
        self.apply_transformation = apply_transformation
        self.target_var = target_var

        # Ensure the run directory exists
        if not os.path.exists(self.run_dir):
            raise FileNotFoundError(f"The specified run directory does not exist: {self.run_dir}")

        # Ensure the CSV directory exists
        if not os.path.exists(self.csv_dir):
            raise FileNotFoundError(f"The specified CSV directory does not exist: {self.csv_dir}")
        
        # Ensure the eval_list directory exists
        if not os.path.exists(self.eval_list):
            raise FileNotFoundError(f"The specified evaluation list directory does not exist: {self.eval_list}")
        
        run_dir = Path("runs/" + self.run_dir)
        if not skip_sim:
            eval_run(run_dir = run_dir, period = "test")
            
        self.result_df: pd.DataFrame = self.__evaluate__(self.eval_list)
        
    
    def __evaluate_single__(self, basin_id: str):
        
        run_dir = Path("runs/" + self.run_dir)

        # set the epoch number
        epoch_num = str(self.epoch_num)
        if len(epoch_num) == 1:
            epoch_num = "model_epoch00" + epoch_num
        elif len(epoch_num) == 2:
            epoch_num = "model_epoch0" + epoch_num
        else:
            epoch_num = "model_epoch" + epoch_num
            
        with open(run_dir / "test" / epoch_num / "test_results.p", "rb") as fp:
            results = pickle.load(fp)
        
         # simulated values
        qsim = results[basin_id]['1D']['xr']['discharge_sim']
        sim = qsim.values
        dates = qsim['date'].values
        
        if self.apply_transformation:
            sim = (sim * np.sqrt(self.var)) + self.mean
            
            sim = np.exp(sim) - 1e-6
            
        csv_file_path = Path(self.csv_dir) / f"{basin_id}.csv"
        df = pd.read_csv(csv_file_path, index_col='date', parse_dates=True).loc[self.test_start_date:self.test_end_date, ['discharge']]
        
        obs = df['discharge'].values
        
        obs_sim_df = pd.DataFrame({
            'date': dates,
            'observed': obs.flatten(),
            'simulated': sim.flatten()
        })
        
        # drop rows with missing values
        obs_sim_df = obs_sim_df.dropna()
        
        # calculate nse
        observed_values = obs_sim_df['observed'].values
        simulated_values = obs_sim_df['simulated'].values
        
        mean_observed = np.mean(observed_values)
        sum_squared_diff = sum((obs - sim) ** 2 for obs, sim in zip(observed_values, simulated_values))
        mean_observed = np.mean(observed_values)
        sum_squared_diff_mean = sum((obs - mean_observed) ** 2 for obs in observed_values)
        nse = None
        
        if sum_squared_diff_mean == 0:
        # Observed data is constant => no variability => NSE is undefined
            nse = np.nan  # or some sentinel value
        else:
            nse = 1 - (sum_squared_diff / sum_squared_diff_mean)
            nse = nse.item()
            
        return nse, observed_values, simulated_values
        
    def __evaluate__(self, eval_list: str):
        
        with open(eval_list, 'r') as file:
            basin_ids = file.read().splitlines()
            
        nse_values = []
        
        for basin_id in tqdm(basin_ids):
            nse, _, _ = self.__evaluate_single__(basin_id)
            nse_values.append(nse)
        
        nse_df = pd.DataFrame({'basin_id': basin_ids, 'NSE': nse_values})
        
        return nse_df